# Data Cleaning & Visualization Project: Walmart Sales Data
This notebook demonstrates how to process raw data and visualize insights.
Key features:
- Handling missing values, outliers, and duplicates.
- Using Pandas, Matplotlib, and Seaborn for data manipulation and visualization.
- Creating visual reports of key findings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set aesthetic parameters for seaborn
sns.set_theme(style="whitegrid", palette="pastel")

In [ ]:
# Load the dataset
df = pd.read_csv('Walmart.csv')
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Handle Missing Values and Duplicates
print("Missing values before cleaning:")
print(df.isnull().sum())

# Drop rows with missing values (or impute depending on context)
df = df.dropna()

# Check and remove duplicates
duplicates = df.duplicated().sum()
print(f"\nNumber of duplicates found: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()

In [ ]:
# Handle Outliers (e.g., in Weekly_Sales)
plt.figure(figsize=(10, 5))
sns.boxplot(x=df['Weekly_Sales'])
plt.title('Boxplot of Weekly Sales before Outlier removal')
plt.show()

# Removing outliers using IQR method for Weekly_Sales
Q1 = df['Weekly_Sales'].quantile(0.25)
Q3 = df['Weekly_Sales'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df = df[(df['Weekly_Sales'] >= lower_bound) & (df['Weekly_Sales'] <= upper_bound)]
print(f"Dataset shape after outlier removal: {df.shape}")

In [ ]:
# Visualization 1: Sales Distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['Weekly_Sales'], bins=30, kde=True, color='skyblue')
plt.title('Distribution of Weekly Sales')
plt.xlabel('Weekly Sales')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Visualization 2: Correlation Heatmap
plt.figure(figsize=(10, 8))
# Drop non-numeric for correlation
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Visualization 3: Average Sales by Holiday Flag
plt.figure(figsize=(8, 5))
sns.barplot(x='Holiday_Flag', y='Weekly_Sales', data=df, estimator=np.mean)
plt.title('Average Weekly Sales: Non-Holiday vs Holiday')
plt.xlabel('Holiday Flag (0 = No, 1 = Yes)')
plt.ylabel('Average Weekly Sales')
plt.show()

## Machine Learning: Predicting High Sales
In this section, we build models to predict outcomes based on the data.
To apply classification evaluation metrics like Confusion Matrices and ROC curves, we'll convert our target variable `Weekly_Sales` into a binary outcome (`High_Sales`), which is 1 if the sales are above the median and 0 otherwise.

We will train and evaluate:
- Logistic Regression (Classification analog to Linear Regression)
- Decision Tree
- Random Forest

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc, classification_report

# Create binary target: 1 if Weekly_Sales > median, else 0
median_sales = df['Weekly_Sales'].median()
df['High_Sales'] = (df['Weekly_Sales'] > median_sales).astype(int)

# Features and Target
features = ['Store', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Holiday_Flag']
X = df[features]
y = df['High_Sales']

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape: {X_test_scaled.shape}")

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

results = {}

# Train and Test Models
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    
    results[name] = {
        'Accuracy': acc,
        'Confusion Matrix': cm,
        'FPR': fpr,
        'TPR': tpr,
        'AUC': roc_auc
    }
    
    print(f"--- {name} ---")
    print(f"Accuracy: {acc:.4f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

In [ ]:
# Visualize Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, metrics) in zip(axes, results.items()):
    sns.heatmap(metrics['Confusion Matrix'], annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f"{name}\nAccuracy: {metrics['Accuracy']:.4f}")
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize ROC Curves
plt.figure(figsize=(10, 8))
for name, metrics in results.items():
    plt.plot(metrics['FPR'], metrics['TPR'], label=f"{name} (AUC = {metrics['AUC']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()